# NRM-Newton Demo

This notebook walks through the full pipeline for finding a robot arm morphology that suits a given manipulation task. The same four steps apply to any task you want to optimise for: Define what the arm needs to reach, run the optimizer, inspect the chosen arm's geometry, and verify that the result is physically sensible. If you want to try a different task, the only thing you normally need to change is the pose set and environment in section 1. Everything else stays the same.

In [1]:
import math
import os
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1  Defining the Task

A task is a list of end-effector poses the arm must reach, paired with an environment that describes the obstacles it must avoid. Here we use ten poses arranged on a half-circle arch in front of a wall, which gives the optimizer a realistic, workspace-limited scenario to work with.

To adapt this for your own use case, replace `create_task` with any function or list that returns SE(3) poses as homogeneous 4×4 tensors, and swap `wall_environment` for your own `Environment` object. The rest of the pipeline treats the task as a black box, so there is no coupling between the task definition and the optimizer.

In [2]:
from core import Task
from tasks.environment import wall_environment
from tasks.sampling.trajectory_pose_sampler import create_task

task_poses = create_task(num_poses=10)
env = wall_environment()
task = Task(environment=env, goal_poses=task_poses)
print(f"Task: {len(task_poses)} goal poses, {len(env.obstacles)} obstacle(s)")

Task: 10 goal poses, 1 obstacle(s)


## 2  Running the Optimization

`optimize_morphology` searches over the space of robot arm designs for your task. It first enumerates all valid discrete joint axis configurations (alpha sequences) for the degree-of-freedom counts you request via `candidate_dof`, scores each one with the NRM reachability surrogate, and then refines the best candidates' continuous link lengths using AdamW gradient descent. The top scorers are handed off to cuRobo for full inverse kinematics validation, and the arm with the best combined reachability and IK success rate is returned.

The `seed_morph` passed in is not a starting point for the search. The optimizer ignores its alpha values and link lengths entirely and generates those from scratch. The only things it reads from `seed_morph` are the device (so it knows where to allocate tensors) and the link radius (the collision sphere size used during cuRobo validation). If you want to run the optimizer on different hardware or with thicker or thinner links, those are the two values to change.

The parameters below are set to a short demo run. For a more thorough search you would raise `num_iterations` to around 500 and `number_random_seed` to 32. The `candidate_dof` field accepts a list such as `[5, 6, 7]` if you only want to consider specific DOF counts, or `"all"` to let the optimizer choose freely. You can also set `ignore_obstacles` to `True` if your task has no collision constraints, which speeds things up considerably.

In [3]:
from core import Morphology
from methods.candidate_selection.static import optimize_morphology

# The seed morphology only carries link_radius and device;
# alpha and link lengths are overridden by the optimizer.
seed_morph = Morphology(
    params=torch.cat([torch.zeros(8, 1), torch.full((8, 2), 0.15)], dim=1).to(device)
)

opt_params = {
    "num_iterations": 50,  # full run uses 500
    "learning_rate": 0.01,
    "logging": True,
    "csv_logging": True,
    "random_seed": 0,
    "number_random_seed": 8,  # full run uses 32
    "percentage_poses": 1,
    "candidate_dof": "all",
    "candidate_batch_size": 96,
    "ignore_ground": True,
    "ignore_obstacles": False,
    "num_plan_candidates": 1,
}

optimized_morph, csv_path, timing, candidates = optimize_morphology(
    morph=seed_morph,
    task=task,
    optimization_parameters=opt_params,
)
print(f"Done. Results written to {csv_path}")

[Info] Starting DOF candidate optimization on device cuda:0.
[Info] dof='all', candidate_dofs=[5, 6, 7], num_iterations=50, learning_rate=0.01, num_alpha_candidates=ALL, candidate_batch_size=96, distribution_batch_size=128, top_probability_fraction=0.025, early_stopping_patience=5, delta_early_stopping=0.0001, random_seed=0, number_random_seed=8, percentage_poses=1.0
[Info] Loading NRM checkpoint: /u/halle/arke/home_at/Documents/nrm-newton/data/weights/checkpoint_5-7.pth
[Info] DOF5 alpha candidates generated: 648 / max_valid=648 (using_all=True).
[Info] DOF6 alpha candidates generated: 1892 / max_valid=1892 (using_all=True).
[Info] DOF7 alpha candidates generated: 5524 / max_valid=5524 (using_all=True).
[Info] Loaded multi-DOF initial candidate morphologies from cache: /u/halle/arke/home_at/Documents/nrm-newton/data/initial_candidates/DOF5-7_seed0/candidates.json (DOF5=631, DOF6=1665, DOF7=4267).
[Info] Writing CSV log to: /u/halle/arke/home_at/Documents/nrm-newton/output/20260816_224

single-round NRM optimization: 100%|██████████| 50/50 [00:01<00:00, 35.19it/s, active=463, batches=5, mean_prob=0.8231, stopped=168, total=631]


[Info] DOF5 early stopping summary: 168/631 candidates stopped before max iteration.


post-checking DOF5 candidate distribution: 100%|██████████| 5/5 [00:00<00:00, 10.44it/s]


[Info] DOF5 post-optimization distribution filter: kept 396/631 candidates.
[Info] NRM optimization tensors: model_device=cuda:0, alpha_device=cuda:0, length_device=cuda:0, task_vec_device=cuda:0, num_candidates=1665, num_poses=10, candidate_batch_size=96, max_candidate_pose_pairs_per_batch=960


...

[Info] DOF7 early stopping summary: 2454/4267 candidates stopped before max iteration.


post-checking DOF7 candidate distribution: 100%|██████████| 34/34 [00:01<00:00, 24.21it/s]


[Info] DOF7 post-optimization distribution filter: kept 2109/4267 candidates.
[Info] Final-link d filter: kept 1088/3521 candidates with processed params[-1, 2] >= 0.
[Info] Writing all-candidates CSV log to: /u/halle/arke/home_at/Documents/nrm-newton/output/20260816_224043/final_candidates.csv
[Info] Top-probability selection: valid_candidates=1088, top_k=28, valid_by_dof={5: 166, 6: 327, 7: 595}, top_by_dof={5: 0, 6: 4, 7: 24}, best_prob=0.999808, worst_top_prob=0.998959


validating top-probability candidates: 100%|██████████| 28/28 [01:04<00:00,  2.30s/it]

[Info] Validation selection: best_ik_success_pose_rate=100.00%, num_best_rate_candidates=5, num_tie_break_candidates=5, final_idx=0, final_dof=7, final_length_sum=1.000000
[Final candidate] dof=7, loss=0.000192, nrm_prob=0.999808, final_se3_err=0.000000, ik_success_pose_rate=100.00%, length_sum=1.000000
Final alpha [deg]:
tensor([-90.,   0.,  90.,  90., -90., -90.,   0., -90.])
Final optimized morphology params:
tensor([[-1.5708, -0.0000,  0.0000],
        [ 0.0000,  0.1590,  0.0000],
        [ 1.5708,  0.0000, -0.1789],
        [ 1.5708,  0.0912,  0.0000],
        [-1.5708, -0.0000,  0.1910],
        [-1.5708, -0.0000,  0.0000],
        [ 0.0000,  0.3800,  0.0000],
        [-1.5708, -0.0000, -0.0000]])
Done. Results written to /u/halle/arke/home_at/Documents/nrm-newton/output/20260816_224043/morphology_history.csv


## 3  The Selected Arm

The result is a `Morphology` object whose `params` tensor holds the Modified Denavit-Hartenberg parameters row by row. Each row corresponds to one link and contains three values: the joint twist angle alpha (in radians), the link length a (in metres), and the link offset d.

If you want to export the arm to another tool, `optimized_morph.params` is a plain PyTorch tensor on the device you specified, so you can call `.cpu().numpy()` on it and write it out in whatever format your downstream code expects.

In [4]:
print(f"Selected arm: DOF {optimized_morph.n_links - 1}")
print()
print(f"  {'Link':>4}  {'α (rad)':>9}  {'α (°)':>7}  {'a (m)':>7}  {'d (m)':>7}")
print("  " + "-" * 46)
for i, (alpha_val, a, d) in enumerate(optimized_morph.params.tolist()):
    alpha_deg = math.degrees(alpha_val)
    print(f"  {i:>4}  {alpha_val:>9.4f}  {alpha_deg:>7.1f}  {a:>7.4f}  {d:>7.4f}")

Selected arm: DOF 7

  Link    α (rad)    α (°)    a (m)    d (m)
  ----------------------------------------------
     0    -1.5708    -90.0  -0.0000   0.0000
     1     0.0000      0.0   0.1590   0.0000
     2     1.5708     90.0   0.0000  -0.1789
     3     1.5708     90.0   0.0912   0.0000
     4    -1.5708    -90.0  -0.0000   0.1910
     5    -1.5708    -90.0  -0.0000   0.0000
     6     0.0000      0.0   0.3800   0.0000
     7    -1.5708    -90.0  -0.0000  -0.0000


## 4  Validating the Selected Arm

Before trusting the optimizer's choice it is worth checking that the arm sits within the region of morphology space the NRM surrogate was trained on, and that it does not collide with itself or show poor manipulability. The distribution checker does both of these things in one call.

A PASS means the arm is a safe input for the NRM model and behaves sensibly in simulation. If you get a FAIL, the `report.reasons` list tells you which checks failed. Common causes are a very unusual alpha configuration that sits outside the training distribution, or link lengths that produce near-singular poses across much of the workspace. In those cases it is usually worth tightening the `candidate_dof` list or adding more iterations so the optimizer has more room to find a better candidate.

In [5]:
from validation.distribution_checker import check_morphology_distribution

report = check_morphology_distribution(optimized_morph)[0]
print("Distribution check:", "PASS" if report.valid else "FAIL")
if not report.valid:
    print("Reasons:", report.reasons)
else:
    print(f"Self-collision rate:  {report.sampled_self_collision_rate:.3f}")
    print(f"Mean Yoshikawa:       {report.mean_yoshikawa:.5f}")

Distribution check: PASS
Self-collision rate:  0.247
Mean Yoshikawa:       0.01019


## 5  IK and FK Validation

The distribution checker in section 4 only looks at the morphology's geometric properties. This section actually runs inverse kinematics and forward kinematics against all ten task poses to measure how accurately the selected arm can reach each one.

The validation builds a cuRobo IK solver for the arm, runs it on every goal pose, then re-applies forward kinematics to the returned joint configurations so we can measure the residual position and rotation error directly in Cartesian space. This is the same validation that runs internally at the end of the optimizer, but here we run it explicitly on the final selected arm so the numbers are easy to inspect.

In [6]:
from validation.optimization_validation import (
    build_optimization_validation_context,
    run_optimization_validation,
)

scene = build_optimization_validation_context(task)
generator = torch.Generator(device=device)
generator.manual_seed(0)

val = run_optimization_validation(
    processed_morphology=optimized_morph.params,
    morph=optimized_morph,
    task=task,
    scene=scene,
    device=device,
    percentage_poses=1.0,
    number_random_seed=32,
    pose_sampling_generator=generator,
)

print(f"IK success rate      : {val['ik_success_pose_rate']:.3f}")
print(f"Mean position error  : {val['best_pos_err_mean']:.4f} m")
print(f"Mean rotation error  : {val['best_rot_err_mean']:.4f} rad")
print(f"Mean SE(3) distance  : {val['best_se3_dist_mean']:.6f}")

IK success rate      : 0.000
Mean position error  : 0.0166 m
Mean rotation error  : 0.0001 rad
Mean SE(3) distance  : 0.005874


## 6  Motion Planning

IK validation confirms that joint configurations exist for each goal pose in isolation, but it does not check whether the arm can actually move between them without hitting anything. This section runs a full collision-aware motion planner using cuRobo's trajectory optimiser, which plans a smooth joint-space path from a feasible start configuration through all ten goal poses in sequence.

The planner samples a collision-free start configuration automatically. If your task has a fixed start joint state you want to use instead, you can pass it as the `start_q` argument to `plan_sequence`. Planning is done sequentially: the joint state at the end of each sub-plan becomes the start of the next, so the full path is continuous.

A successful result means the arm can reach every goal in order without self-collision or obstacle penetration. If planning fails at one of the goals, `result.failed_at_goal` tells you which one and `result.path` contains the partial trajectory up to that point, which is useful for diagnosing whether the problem is a locally cluttered configuration or a globally unreachable pose.

In [7]:
from planning.curobo_planner import CuroboPlanner

planner = CuroboPlanner(
    optimized_morph,
    task,
    device,
    ignore_ground=opt_params["ignore_ground"],
    ignore_obstacles=opt_params["ignore_obstacles"],
)
start_q = planner.default_start_q().to(optimized_morph.params.dtype)

if not planner.is_q_feasible(start_q):
    print("Sampled start configuration is in collision — re-run the cell to try again.")
else:
    print(f"Start configuration: {start_q.tolist()}")
    result, final_q = planner.plan_sequence(task.goal_poses, start_q)

    if result.success:
        print(
            f"\nPlanning succeeded: {len(result.path)} waypoints through {task.goal_poses.shape[0]} goals."
        )
    else:
        print(f"\nPlanning failed at goal {result.failed_at_goal}.")
        if result.path:
            print(f"  Partial path: {len(result.path)} waypoints.")

[Info] Feasible start config found (sample 2/4096).
Start configuration: [2.0122337341308594, 3.1833252906799316, 2.573514461517334, 2.757418632507324, -2.668956756591797, 4.1901021003723145, 2.8906893730163574]


Start or End state in collision
Start or End state in collision
Start or End state in collision


[cuRobo] Goal 9/10 failed (90.0% reached): trajectory
[cuRobo]   pose OK (0.5mm/0.001rad), goal 9 plan waypoint 35/101: world collision (box_0 ↔ link_4, pen 0.001 m)

Planning failed at goal 9.
  Partial path: 449 waypoints.
